In [1]:
import numpy as np
import pandas as pd
import gc
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.decomposition import PCA
import datetime
from datetime import timedelta
import random
from imblearn.over_sampling import RandomOverSampler
from numba import njit
import numba as nb
from scipy.stats import zscore
import json
from scipy.stats import spearmanr
import websocket
import warnings
warnings.filterwarnings('ignore')
import random as python_random
import statsmodels.api as sm
import joblib
import sys
import time
import traceback
import torch
import torch.nn as nn
import torch.nn.functional as F 
import os
import importlib
TL = importlib.import_module("torch")
NN = TL.nn
F = importlib.import_module("torch.nn.functional")

from scipy.stats import spearmanr
np.random.seed(150)
python_random.seed(150)
import itertools
import copy

In [2]:
hold_pos = 20

In [3]:
#@njit
def run_rolling_nis_dynamic_numba(log_prices, periods, q_lookback, Q_scale=0.1, R_scale=1.0):
    n = len(log_prices)
    sum_nis_output = np.empty(n, dtype=np.float64)
    sum_nis_output[:] = np.nan
    n2 = periods - 1

    # O loop começa garantindo que temos histórico suficiente para o q_lookback e para a janela de teste
    start_idx = max(n2, q_lookback)
    

    # Loop Externo: Desliza a janela de teste ao longo do tempo
    for i in range(start_idx, n):
        window_start = i - n2
        
        # 1. CÁLCULO DINÂMICO: Extrai o histórico estritamente anterior à janela atual
        # Ex: Se q_lookback=120, olha os 120 retornos passados para calibrar o momento atual
        hist_start = window_start - q_lookback
        if hist_start < 0:
            hist_start = 0
            
        hist_prices = log_prices[hist_start : window_start + 1]
        
        # Calcula retornos e variância dinamicamente para esta barra específica
        if len(hist_prices) > 1:
            hist_returns = hist_prices[1:] - hist_prices[:-1]
            base_variance = np.var(hist_returns)
        else:
            base_variance = 1e-8

        # Se o mercado congelar/平, evita divisão por zero
        if base_variance <= 0.0: 
            base_variance = 1e-8

        # 2. ATUALIZAÇÃO DINÂMICA DE Q E R
        global_q = base_variance * Q_scale
        global_r = base_variance * R_scale

        # Inicialização do Filtro para a janela atual com os parâmetros novos
        state_estimate = log_prices[window_start]
        P = global_r 
        window_sum_nis = 0.0

        # Loop Interno: Roda o Filtro de Kalman local com o Q e R atualizados
        for j in range(1, periods):
            price_t = log_prices[window_start + j]

            # Predição
            pred_state = state_estimate
            pred_P = P + global_q

            # Atualização
            innovation = price_t - pred_state
            S_t = pred_P + global_r

            # Acúmulo do NIS
            nis_t = (innovation ** 2) / S_t
            window_sum_nis += nis_t

            # Atualização dos Estados
            K_t = pred_P / S_t
            state_estimate = pred_state + K_t * innovation
            P = (1.0 - K_t) * pred_P

        sum_nis_output[i] = window_sum_nis

    return sum_nis_output

In [4]:
def scan_dataframe_fast_dynamic(df, periods=90, q_lookback=180):
    log_prices_arr = df['Log'].to_numpy()
    

    # Executa a função dinâmica (Q e R mudam a cada linha do dataframe)
    sum_nis_array = run_rolling_nis_dynamic_numba(
        log_prices_arr, periods, q_lookback, Q_scale=0.01, R_scale=10.0
    )

    # Wilson-Hilferty para Chi-Squared 95%
    df_degrees = periods - 1
    z_90 = 1.2815515655446004
    z_95 = 1.6448536269514722
    z_99 = 2.326
    critical_value = df_degrees * (1.0 - 2.0 / (9.0 * df_degrees) + z_95 * np.sqrt(2.0 / (9.0 * df_degrees))) ** 3

    df['sum_nis'] = sum_nis_array
    df['is_stationary'] = sum_nis_array <= critical_value
    df['market_regime'] = np.where(
        df['sum_nis'].isna(), 'WARMUP',
        np.where(df['is_stationary'], 'STATIONARY', 'NON-STATIONARY')
    )
    
    return df

In [5]:
class SwiGLULayer(NN.Module):
    def __init__(self, in_f, out_f, drop):
        super().__init__()
        self.dense = NN.Linear(in_f, out_f*2, bias=False)
        self.drop = NN.Dropout(drop)
    def forward(self, x):
        g, v = TL.chunk(self.dense(x), 2, dim=-1)
        return self.drop(F.silu(g) * v)
class SwiGLUBaseModel(NN.Module):
    def __init__(self, in_dim, hid, drop):
        super().__init__()
        self.ln_in = NN.LayerNorm(in_dim)
        self.l1 = SwiGLULayer(in_dim, hid, drop)
        self.ln1 = NN.LayerNorm(hid)
        self.l2 = SwiGLULayer(hid, hid, drop)
        self.ln2 = NN.LayerNorm(hid)
        self.out = NN.Linear(hid, 1)
    def forward(self, x):
        x = self.ln_in(x)
        x = self.l1(x)
        x = self.ln1(x)
        x = self.l2(x)
        x = self.ln2(x)
        return self.out(x)

In [6]:
# Load your 0.1910 model - 10 features, 256 hidden
device = TL.device('cuda' if TL.cuda.is_available() else 'cpu')
model = SwiGLUBaseModel(11, 256, 0.0).to(device)  # 10 features, 256 hidden - from your last run
model.load_state_dict(TL.load('best_11f_256_1868.pt', map_location=device))
model.eval()

SwiGLUBaseModel(
  (ln_in): LayerNorm((11,), eps=1e-05, elementwise_affine=True, bias=True)
  (l1): SwiGLULayer(
    (dense): Linear(in_features=11, out_features=512, bias=False)
    (drop): Dropout(p=0.0, inplace=False)
  )
  (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (l2): SwiGLULayer(
    (dense): Linear(in_features=256, out_features=512, bias=False)
    (drop): Dropout(p=0.0, inplace=False)
  )
  (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (out): Linear(in_features=256, out_features=1, bias=True)
)

In [7]:
@njit
def fast_poc_expanding_numba(close, volume, max_window=300, min_periods=70, bins=20):
    n = len(close)
    out = np.full(n, np.nan)

    # Começamos a calcular apenas após atingir o min_periods (70)
    for i in range(min_periods - 1, n):
        # Lógica da Janela Expansiva:
        # Se i < 300, a janela é [0 : i+1] (tamanho i+1)
        # Se i >= 300, a janela é [i-299 : i+1] (tamanho fixo 300)
        start = max(0, i - max_window + 1)

        w_close = close[start : i + 1]
        w_vol = volume[start : i + 1]

        p_min = np.min(w_close)
        p_max = np.max(w_close)

        if p_min >= p_max:
            out[i] = 0.0
            continue

        # Histograma manual (Otimizado para Numba)
        hist = np.zeros(bins)
        bin_width = (p_max - p_min) / bins

        for j in range(len(w_close)):
            # Calcula o bin correspondente
            val = w_close[j]
            idx = int((val - p_min) / bin_width)
            if idx >= bins: 
                idx = bins - 1 # Garante que o valor máximo entre no último bin
            hist[idx] += w_vol[j]

        # Encontra o bin com maior volume (Point of Control)
        max_bin_idx = np.argmax(hist)
        poc = p_min + (max_bin_idx + 0.5) * bin_width

        # p_last / poc - 1 (Sinal de desvio do preço médio de volume)
        out[i] = close[i] / poc - 1

    return out

# call function
def VPRO2(df,  max_window=300, min_periods=70, bins=20):
    df['Vpro_20'] = fast_poc_expanding_numba(
        df['close'].values, 
        df['volume'].values, 
        max_window=300, 
        min_periods=70,
        bins=20
    )
    return df

############### AARON #######################################
def ARO2(df):

    @njit
    def fast_aroon_osc(high, low, window):
        n = len(high)
        out = np.full(n, np.nan)

        for i in range(window - 1, n):
            max_idx = 0
            min_idx = 0
            max_val = high[i - window + 1]
            min_val = low[i - window + 1]

            for j in range(1, window):
                curr_h = high[i - window + 1 + j]
                curr_l = low[i - window + 1 + j]

                if curr_h >= max_val:
                    max_val = curr_h
                    max_idx = j
                if curr_l <= min_val:
                    min_val = curr_l
                    min_idx = j

            days_since_max = (window - 1) - max_idx
            days_since_min = (window - 1) - min_idx

            up = ((window - days_since_max) / window) * 100
            down = ((window - days_since_min) / window) * 100
            
            out[i] = up - down
            
        return out

    # Direct extraction and single function execution for window = 15
    df['AAR_195'] = fast_aroon_osc(df['high'].values, df['low'].values, 195)
    
    return df
##################################################################

###### Weighted MACD short term ######


@nb.njit(cache=True)
def ewm_numba(arr, span):
    """Calculates Exponential Moving Average matching Pandas 'adjust=False'."""
    n = len(arr)
    out = np.empty(n, dtype=np.float64) # 🟢 FIXED: Allocates a clean, isolated array in memory
    if n == 0:
        return out

    alpha = 2.0 / (span + 1.0)

    # Safely prime the first valid element
    if n > 0:
        out[0] = arr[0]

    for i in range(1, n):
        if np.isnan(arr[i]):
            out[i] = out[i-1]
        elif np.isnan(out[i-1]):
            out[i] = arr[i]
        else:
            out[i] = arr[i] * alpha + out[i-1] * (1.0 - alpha)
    return out

@nb.njit(cache=True)
def rolling_mean_numba(arr, window):
    """Calculates standard Simple Moving Average (SMA)."""
    n = len(arr)
    out = np.empty(n)
    out[:] = np.nan

    if n < window:
        return out

    current_sum = 0.0
    for i in range(window):
        current_sum += arr[i]
    out[window - 1] = current_sum / window

    for i in range(window, n):
        current_sum += arr[i] - arr[i - window]
        out[i] = current_sum / window
    return out

@nb.njit(parallel=True, cache=True)
def process_signals_numba(close,n1, n2, n3):
    """Processes MACD EWM/Rolling layers and handles vectorized signal logic."""
    n = len(close)

    # --- 1. EXPONENTIAL MACD LAYER (EWM) ---
    ma_fast_ewm = ewm_numba(close, n1)
    ma_slow_ewm = ewm_numba(close, n2)
    ma_ewm = ma_fast_ewm - ma_slow_ewm
    sig_ewm = ewm_numba(ma_ewm, n3)
    macds_ewm = ma_ewm - sig_ewm
    macd_out = np.zeros(n)

    # --- 2. ROLLING MACD LAYER (SMA) ---
    ma_fast_roll = rolling_mean_numba(close, n1)
    ma_slow_roll = rolling_mean_numba(close, n2)
    ma_roll = ma_fast_roll - ma_slow_roll
    sig_roll = ewm_numba(ma_roll, n3)
    macds_roll = ma_roll - sig_roll
    macdn_out = np.zeros(n)

    
    return macd_out, ma_ewm

# --- 2. MAIN INTEGRATION WRAPPER FOR PANDAS DATAFRAMES ---

def execute_strategy_numba(df):
    """Extracts NumPy arrays from DataFrame, triggers Numba, and commits back."""
    close = np.ascontiguousarray(df['close'].to_numpy(dtype=np.float64))
    low = np.ascontiguousarray(df['low'].to_numpy(dtype=np.float64))
    high = np.ascontiguousarray(df['high'].to_numpy(dtype=np.float64))
    volume = np.ascontiguousarray(df['volume'].to_numpy(dtype=np.float64))


    # Step 2:  execution
    i=84
    n1, n2, n3 = i, 2*i, int(i/1.5) # 🟢 OPTIMIZED: Forces explicit int signature to ensure clean compilation x, 2x, x/1.5

    macd_out, ma_ewm = process_signals_numba(
        close, n1, n2, n3
    )

    #df[f'MACD_{i}'] = macd_out
    df[f'MA_{i}'] = ma_ewm
    #df[f'MACDS_{i}'] = macds_ewm
    #df[f'MACDN_{i}'] = macdn_out

    return df

def WMA2(df):
    # Execute the strategy using your DataFrame
    df = execute_strategy_numba(df)
    return df

###################### VWAP ############################################

#@njit
def fast_vwap_zscore(close, high, low, volume, max_window=300, min_periods=30):
    n = len(close)
    vwap_z = np.full(n, np.nan)

    # Preço Típico ponderado por Volume: (H+L+C)/3 * Vol
    tp_vol = ((close + high + low) / 3.0) * volume

    for i in range(min_periods - 1, n):
        # Define o início da janela (expansiva até max_window)
        start = max(0, i - max_window + 1)

        # Somas para o VWAP
        sum_tp_vol = np.sum(tp_vol[start : i + 1])
        sum_vol = np.sum(volume[start : i + 1])

        if sum_vol == 0:
            vwap_z[i] = 0.0
            continue

        current_vwap = sum_tp_vol / sum_vol

        # Para o Z-Score (df['Close'] - vwap) / std(vwap)
        # Note: Seu código original calcula o desvio padrão do HISTÓRICO do VWAP
        if i >= start + min_periods:
            # Precisamos calcular o desvio padrão dos valores de VWAP na janela
            # Para performance, vamos extrair o histórico de VWAPs parciais
            window_vwaps = np.zeros(i - start + 1)
            temp_tp_vol = 0.0
            temp_vol = 0.0

            # Este sub-loop calcula o histórico de VWAPs para o desvio padrão
            for k in range(start, i + 1):
                temp_tp_vol = np.sum(tp_vol[start : k + 1])
                temp_vol = np.sum(volume[start : k + 1])
                window_vwaps[k - start] = temp_tp_vol / temp_vol if temp_vol != 0 else current_vwap

            std_vwap = np.std(window_vwaps)

            if std_vwap != 0:
                vwap_z[i] = (close[i] - current_vwap) / std_vwap
            else:
                vwap_z[i] = 0.0

    return vwap_z

def VWAP2(df):
    df['VWAP_300'] = fast_vwap_zscore(
        df['close'].values, df['high'].values, 
        df['low'].values, df['volume'].values, 300
    )
    return df
    
########################## CCI #############################
    
def CCI2(df):

    #@njit
    def calcular_cci_fast(typ_price, window):
        n = len(typ_price)
        out = np.full(n, np.nan)

        for i in range(window - 1, n):
            # 1. Select window Tipical Price
            janela = typ_price[i - window + 1 : i + 1]

            # 2. MA Tipical price (TP_MA)
            tp_ma = np.mean(janela)

            # 3.Absolute Deviation (MAD) 
            # 
            mad = np.mean(np.abs(janela - tp_ma))

            # 4. Cálculate do CCI
            if mad != 0:
                out[i] = (typ_price[i] - tp_ma) / (0.015 * mad)
            else:
                out[i] = 0

        return out

    # Integration on the Loop
    tp = ((df['high'] + df['low'] + df['close']) / 3).values

    i = 150
    df[f'CCI_{i}'] = calcular_cci_fast(tp, i)

    return df

############### RSI ##########################

def RSI2(df):
    df['S1_pct'] = df['close'].pct_change()

    #@njit
    def calcular_rsi_wilder(changes, window):
        n = len(changes)
        rsi = np.full(n, np.nan)

        # Separate gains and losses (Vetorization Numba)
        gains = np.where(changes > 0, changes, 0.0)
        losses = np.where(changes < 0, -changes, 0.0)

        # Inicialization (Simple avg for the first value)
        avg_gain = np.mean(gains[:window])
        avg_loss = np.mean(losses[:window])

        if avg_loss == 0:
            rsi[window-1] = 100
        else:
            rs = avg_gain / (avg_loss + 1e-10)
            rsi[window-1] = 100 - (100 / (1 + rs))

        # Wilder softness (Recursive)
        for i in range(window, n):
            avg_gain = (avg_gain * (window - 1) + gains[i]) / window
            avg_loss = (avg_loss * (window - 1) + losses[i]) / window

            if avg_loss == 0:
                rsi[i] = 100
            else:
                rs = avg_gain / (avg_loss + 1e-10)
                rsi[i] = 100 - (100 / (1 + rs))

        return rsi - 50 # Retorna centralizado no 0 conforme seu código

    # Loop integration
    # S1_pct pct change of Close
    changes = df['S1_pct'].values

    i = 160
    df[f'RSI_{i}'] = calcular_rsi_wilder(changes, i)
        
    return df
        
    

########################## HA ####################################

def HA2(df):

    #@njit # Just-In-Time compilation for max performance 
    def calcular_ha_core(op, hi, lo, cl):
        n = len(op)
        ha_open = np.empty(n)
        ha_close = (op + hi + lo + cl) / 4.0

        # First candle (Seed)
        ha_open[0] = (op[0] + cl[0]) / 2.0

        # Loop recursive 
        for i in range(1, n):
            ha_open[i] = (ha_open[i-1] + ha_close[i-1]) / 2.0

        # High and low
        ha_high = np.maximum(hi, np.maximum(ha_open, ha_close))
        ha_low = np.minimum(lo, np.minimum(ha_open, ha_close))

        # Vetorization of signals (1 for up, -1 for down, 0 neutral)
        sinal = np.zeros(n)
        for i in range(n):
            if (ha_close[i] > ha_open[i]) and (ha_open[i]==ha_low[i]):
                sinal[i] = 1.0
            elif (ha_close[i] > ha_open[i]) and (ha_close[i-1] < ha_open[i]):
                sinal[i] = 1.0
            
            elif (ha_close[i] < ha_open[i]) and (ha_open[i]==ha_high[i]):
                sinal[i] = -1.0  
            elif (ha_close[i] < ha_open[i]) and (ha_close[i-1] > ha_open[i]):
                sinal[i] = -1.0

        return sinal

    def sequence_counter(sinal):
        # Lógic to count sequence (Reset on 0)
        # Convert to NumPy to avoid overhead groups Pandas
        n = len(sinal)
        ha_count = np.zeros(n)
        current_sum = 0.0

        for i in range(n):
            if sinal[i] == 0:
                current_sum = 0.0
            elif i > 0 and sinal[i] != sinal[i-1]:
                current_sum = sinal[i] # Reset e começa nova contagem
            else:
                current_sum += sinal[i]
            ha_count[i] = current_sum
        return ha_count

    # Integration on DataFrame
    df['HA_Sinal'] = calcular_ha_core(df['open'].values, df['high'].values, 
                                      df['low'].values, df['close'].values)

    df['HA_Seq'] = sequence_counter(df['HA_Sinal'].values)
        
        
    return df

 ###############################  Shadow_MA  ##################################################### 
    
@njit
def calc_sma_numba(arr, period):
    n = arr.shape[0]
    out = np.full(n, np.nan, dtype=np.float64)

    if n >= period:
        current_sum = 0.0
        for i in range(period):
            current_sum += arr[i]
        out[period - 1] = current_sum / period

        for i in range(period, n):
            current_sum = current_sum - arr[i - period] + arr[i]
            out[i] = current_sum / period
    return out

# 2. FUNÇÃO PRINCIPAL: Calcula as dinâmicas brutas de candle
@njit
def calc_candle_raw_numba(open_arr, high_arr, low_arr, close_arr):
    n = open_arr.shape[0]
    upper_s = np.empty(n, dtype=np.float64)
    lower_s = np.empty(n, dtype=np.float64)
    body_tr = np.empty(n, dtype=np.float64)

    for i in range(n):
        candle_range = (high_arr[i] - low_arr[i]) + 1e-8
        max_body = open_arr[i] if open_arr[i] > close_arr[i] else close_arr[i]
        min_body = close_arr[i] if open_arr[i] > close_arr[i] else open_arr[i]

        upper_s[i] = (high_arr[i] - max_body) / candle_range
        lower_s[i] = (min_body - low_arr[i]) / candle_range
        body_tr[i] = abs(close_arr[i] - open_arr[i]) / candle_range

    return upper_s, lower_s, body_tr

def SHADOW(df):

    # --- PREPARAÇÃO DOS DADOS ---
    o = df['open'].values.astype(np.float64)
    h = df['high'].values.astype(np.float64)
    l = df['low'].values.astype(np.float64)
    c = df['close'].values.astype(np.float64)

    # Passo 1: Calcula o básico fixo uma única vez (Economiza muito processamento!)
    up_s_raw, lw_s_raw, b_tr_raw = calc_candle_raw_numba(o, h, l, c)

    # Salva as brutas de 1 minuto no DataFrame
    df['Upper_Shadow_Pct'] = up_s_raw
    df['Lower_Shadow_Pct'] = lw_s_raw
    df['Body_to_Range'] = b_tr_raw

    # --- O SEU FOR LOOP PARA DIVERSAS JANELAS ---
    # O range(15, 61, 15) vai testar as janelas: 15, 30, 45, 60
    JANELA_SUAVIZACAO1 = 150
    JANELA_SUAVIZACAO2 = 165
    # Calcula a média móvel no Numba de forma instantânea para a janela atual
    df[f'Upper_Shadow_MA{JANELA_SUAVIZACAO1}'] = calc_sma_numba(up_s_raw, JANELA_SUAVIZACAO1)
    df[f'Lower_Shadow_MA{JANELA_SUAVIZACAO2}'] = calc_sma_numba(lw_s_raw, JANELA_SUAVIZACAO2)


    #df = df.dropna()
    #df = df.reset_index(drop=True)

    return df

##################### TGT ########################

def TGT(df, hold_pos=hold_pos): 
    
    #target_price = df['Avg'].shift(-hold_pos).ewm(span=int(hold_pos), adjust=False).mean()
    #target_price = df['avg'].shift(-hold_pos).rolling(int(hold_pos/2)).mean()
    #df['Tgt'] = (target_price.values / df['avg'].values) - 1

    target_price1 = df['avg'].shift(int(-hold_pos))
    #target_price2 = df['Avg'].shift(int(-hold_pos*0.9))
    #target_price3 = df['Avg'].shift(int(-hold_pos*0.8))
    #target_price4 = df['Avg'].shift(int(-hold_pos*0.7))
    #target_price5 = df['Avg'].shift(int(-hold_pos*0.6))
    #target_price6 = df['Avg'].shift(int(-hold_pos*0.5))

    # 2. Realize a operação usando valores brutos (numpy) para economizar RAM
    df["ret1"] = (target_price1.values / df['avg'].values) - 1
    #df["ret2"] = (target_price2.values / df['Avg'].values) - 1
    #df["ret3"] = (target_price3.values / df['Avg'].values) - 1
    #df["ret4"] = (target_price4.values / df['Avg'].values) - 1
    #df["ret5"] = (target_price5.values / df['Avg'].values) - 1
    #df["ret6"] = (target_price6.values / df['Avg'].values) - 1

    #df['Tgt'] =(df['ret1'] + df['ret2'] + df['ret3'] + df['ret4'] + df['ret5'] + df['ret6'])/6
    #df['Tgt'] =(df['Tgt'] + df['ret1'] +  df['ret4'])/3
    #df['Tgt'] = df['ret1']



    # Define a historical lookback window (e.g., 500 or 1000 rows)
    #z_lookback = 200  

    # Calculate rolling statistical baselines based ONLY on past data
    #rolling_mean = df['Tgt'].rolling(window=z_lookback, min_periods=100).mean().shift(hold_pos)
    #rolling_std = df['Tgt'].rolling(window=z_lookback, min_periods=100).std().shift(hold_pos)
    #df["Mean"] = rolling_mean
    #df["Std"] = rolling_std

    # Apply the Z-score transformation: (X - Mean) / Std
    # We add a tiny epsilon (1e-8) to prevent zero-division errors if volatility crushes to 0
    #df['Tgt'] = (df['Tgt'] - rolling_mean) / (rolling_std + 1e-8)

     #inconcistency
    #df['Tgt'] = np.where((df['Tgt'] > 0) & (df['ret1'] > 0) | 
    #                     (df['Tgt'] < 0) & (df['ret1'] < 0), df['Tgt'], 0)





    df['Trig'] = df['ret1'].abs().rolling(25).mean()         
    #df['Trig'] = df['ret'].abs().rolling(5).mean() 
    #move trig ahead not to use future data
    df['Trig'] = df['Trig'].shift(hold_pos)

    return df

In [ ]:
# ==============================================================================
# STEP 1: PRE-EMPTIVE CLEANUP OF LINGERING MEMORY SOCKETS
# ==============================================================================
if 'ws' in globals():
    try:
        print("Wiping old lingering WebSocket connection from memory...")
        ws.close()
        del ws
    except Exception:
        pass

# ==============================================================================
# STEP 2: BLINDED LIFECYCLE WRAPPER
# ==============================================================================
def start_pipeline_stream():
    global ws
    try:
        print("Initializing clean WebSocket connection tunnel...")
    except KeyboardInterrupt:
        print("\nPipeline execution stopped manually by user.")
    except Exception as e:
        print(f"Critical Pipeline Execution Crash: {str(e)}")
    finally:
        print("Executing emergency resource cleanup. Unlinking network sockets...")
        try:
            ws.close()
        except NameError:
            pass
        print("Success: System state restored. Ready for re-run without kernel restart!")

start_pipeline_stream()

# ==========================================
# 1. LIVE DATA STATE CONTAINER
# ==========================================
class TradingState:
    def __init__(self):
        self.df_acc = pd.DataFrame(columns=['symbol', 'open', 'high', 'low', 'close', 'volume', 't'])
        self.df_acc.index.name = 'timestamp'
        self.df_final = pd.DataFrame(columns=['symbol', 'open', 'high', 'low', 'close', 'volume', 'avg',
                                             'my_p', 'sig1', 'sig2', 'pnl', 'fee', 'real_re', 'trade_pr',
                                              'pos_count'])
        self.df_final.index.name = 'timestamp'
        self.df_final_lim = pd.DataFrame()
        self.count = 0
        self.next_sig = "zero"
        self.realized_re = 0
        self.ntrades = 0
        # FIX 1: trade ledger that is NEVER truncated by tail(200).
        # df_final is cut to 200 rows, so summing pnl over df_final breaks
        # buy/sell pairing. trade_log keeps every fill forever.
        self.trade_log = []

state = TradingState()

device = TL.device('cuda' if TL.cuda.is_available() else 'cpu')
model = SwiGLUBaseModel(11, 256, 0.0).to(device)
model.load_state_dict(TL.load('best_11f_256_1868.pt', map_location=device))
model.eval()

try:
    my_production_scaler = joblib.load('my_production_scaler_ETH_reg.pkl')
    print("### Production Scaler successfully loaded into memory! ###", flush=True)
except FileNotFoundError:
    print("### ERROR: 'my_production_scaler.pkl' or PCA not found! Please export it from your training environment. ###", flush=True)


def process_pipeline_sync(source_data):
    global state, model
    max_time = 50
    min_time = 3
    max_pos = 3
    up = 10.6
    up2 = 0
    do = -10.6
    do2 = 0
    mult_up = 0.7
    mult_do = 0.635
    k = 4
    k_stop = -0.5
    hold_pos = 20
    price1 = 2492.76
    start_window = 200
    window = 200
    l = window + 300

    try:
        payload = json.loads(source_data)
        if 'data' in payload:
            payload = payload['data']
        if 'result' in payload or ('id' in payload and 'k' not in payload):
            return
        if 'k' not in payload:
            return

        kline = payload['k']
        symbol = kline['s']
        open_p = float(kline['o'])
        high_p = float(kline['h'])
        low_p = float(kline['l'])
        close_p = float(kline['c'])
        volume = float(kline['v'])
        t_timestamp = kline['t']
        event_time = pd.to_datetime(payload.get('E', t_timestamp), unit='ms')

        tick_data = {
            'symbol': [str(symbol)], 'open': [open_p], 'high': [high_p],
            'low': [low_p], 'close': [close_p], 'volume': [volume], 't': [int(t_timestamp)]
        }
        df_tick = pd.DataFrame(tick_data, index=[event_time])
        df_tick.index.name = 'timestamp'

        state.df_acc = pd.concat([state.df_acc, df_tick], ignore_index=False)
        if len(state.df_acc) > window:
            state.df_acc = state.df_acc.tail(window)

        df_acc_1 = state.df_acc[state.df_acc['symbol'] == 'ETHUSDT'].copy()
        if df_acc_1.empty:
            return

        df_acc_1['dif'] = df_acc_1['t'].diff()
        df_acc_1['dif2'] = df_acc_1['dif'].shift(-1)
        if not df_acc_1.empty:
            df_acc_1.iat[0, df_acc_1.columns.get_loc('dif2')] = 0.0
        df_acc_1['avg'] = (df_acc_1['open'] + df_acc_1['close'] + df_acc_1['high'] + df_acc_1['low']) / 4.0

        if not kline['x']:
            return

        state.count += 1
        # FIX 2: use candle OPEN time (kline t), NOT event time (E).
        # Event time can duplicate / mismatch and .loc[] then inserts a new
        # NaN row that becomes the last row (sum_nis NaN, y_pred NaN).
        candle_time = pd.to_datetime(kline['t'], unit='ms')

        if not state.df_final.empty:
            state.df_final.index = pd.to_datetime(state.df_final.index, errors='coerce')
            state.df_final = state.df_final[state.df_final.index.notna()]

        live_candle = df_acc_1.iloc[-1]
        if pd.isna(live_candle['open']) or pd.isna(live_candle['close']):
            data_source = df_acc_1.iloc[-2]
        else:
            data_source = live_candle

        candle_time_valid = pd.to_datetime(candle_time)

        # FIX 3: recompute avg from the kline itself, do not trust tick avg.
        # If tick avg is NaN, Log = NaN and sum_nis[-1] = NaN forever.
        avg_val = (open_p + high_p + low_p + close_p) / 4.0
        new_candle_df = pd.DataFrame({
            'symbol': [str(data_source['symbol'])],
            'open': [float(open_p)],
            'high': [float(high_p)],
            'low': [float(low_p)],
            'close': [float(close_p)],
            'volume': [float(volume)],
            'avg': [float(avg_val)],
            'my_p': [0.0],
            'sig1': [0.0],
            'sig2': [0.0], 'pnl': [0.0],
            'fee': [0.0], 'real_re': [0.0],
            'trade_pr': [0.0],
            'pos_count': [0.0]
        }, index=[candle_time_valid])
        new_candle_df.index.name = 'timestamp'

        if len(state.df_final) >= 2:
            last_timestamp = state.df_final.index[-1]

            if state.next_sig == 'long_1':
                trade_price_ref = state.df_final['avg'].at[last_timestamp]
                state.df_final.at[last_timestamp, 'sig1'] = 1
                state.df_final.at[last_timestamp, 'my_p'] = 1
                state.df_final.at[last_timestamp, 'pos_count'] = 1
                state.df_final.at[last_timestamp, 'trade_pr'] = trade_price_ref

            elif state.next_sig == 'short_1':
                trade_price_ref = state.df_final['avg'].at[last_timestamp]
                state.df_final.at[last_timestamp, 'sig1'] = -1
                state.df_final.at[last_timestamp, 'my_p'] = -1
                state.df_final.at[last_timestamp, 'pos_count'] = 1
                state.df_final.at[last_timestamp, 'trade_pr'] = trade_price_ref

            elif state.next_sig == 'long_2':
                trade_price_ref = state.df_final['avg'].at[last_timestamp]
                pos_ref = state.df_final['my_p'].ffill().iloc[-2]
                state.df_final.at[last_timestamp, 'sig1'] = 1
                state.df_final.at[last_timestamp, 'my_p'] = int(pos_ref + 1)
                state.df_final.at[last_timestamp, 'pos_count'] = 1
                state.df_final.at[last_timestamp, 'trade_pr'] = trade_price_ref

            elif state.next_sig == 'close':
                trade_price_ref = state.df_final['avg'].at[last_timestamp]
                pos_ref = state.df_final['my_p'].ffill().iloc[-2]
                state.df_final.at[last_timestamp, 'sig1'] = -pos_ref
                state.df_final.at[last_timestamp, 'my_p'] = 0
                state.df_final.at[last_timestamp, 'pos_count'] = 0
                state.df_final.at[last_timestamp, 'trade_pr'] = trade_price_ref

            elif state.next_sig == 'hold':
                count_ref = state.df_final['pos_count'].ffill().iloc[-2]
                pos_ref = state.df_final['my_p'].ffill().iloc[-2]
                state.df_final.at[last_timestamp, 'sig1'] = 0
                state.df_final.at[last_timestamp, 'my_p'] = pos_ref
                state.df_final.at[last_timestamp, 'pos_count'] = int(count_ref + 1)
                state.df_final.at[last_timestamp, 'trade_pr'] = 0

            elif state.next_sig == 'short_2':
                trade_price_ref = state.df_final['avg'].at[last_timestamp]
                pos_ref = state.df_final['my_p'].ffill().iloc[-2]
                state.df_final.at[last_timestamp, 'sig1'] = -1
                state.df_final.at[last_timestamp, 'my_p'] = int(pos_ref - 1)
                state.df_final.at[last_timestamp, 'pos_count'] = 1
                state.df_final.at[last_timestamp, 'trade_pr'] = trade_price_ref

            elif state.next_sig == 'zero':
                state.df_final.at[last_timestamp, 'sig1'] = 0
                state.df_final.at[last_timestamp, 'my_p'] = 0
                state.df_final.at[last_timestamp, 'pos_count'] = 0
                state.df_final.at[last_timestamp, 'trade_pr'] = 0

            # FIX 4 (was the fatal bug): capture block DEDENTED to the level of
            # "if state.next_sig", OUTSIDE all elif branches.
            # Before it was inside "zero", where sig1 is always 0, so trade_log stayed empty.
            _sig1_written = float(state.df_final.at[last_timestamp, 'sig1'])
            _trade_pr_written = float(state.df_final.at[last_timestamp, 'trade_pr'])
            if _sig1_written != 0:
                state.trade_log.append((_sig1_written, _trade_pr_written))
                print(f"[TRADE LOG] sig1={_sig1_written} trade_pr={_trade_pr_written:.2f} log_len={len(state.trade_log)} net_sig1={sum(s for s,_ in state.trade_log)}")

        if not state.df_final.empty:
            state.df_final = state.df_final.loc[:, ~state.df_final.columns.duplicated()].copy()

        state.df_final = pd.concat([state.df_final, new_candle_df], axis=0, join='outer')
        state.df_final = state.df_final.dropna(subset=['open', 'close', 'avg'])
        state.df_final = state.df_final[~state.df_final.index.duplicated(keep='last')]
        state.df_final = state.df_final.sort_index()
        state.df_final.index.name = 'timestamp'

        if len(state.df_final) > window:
            state.df_final = state.df_final.tail(window)

        print(f"\n[CONFIRMED] CANDLE SAVED AT: {candle_time_valid} | Rows: {len(state.df_final)}")
        state.df_final['pct_return'] = ((state.df_final['close'].astype(float) - price1) / price1) * 100

        MINIMUM_CANDLES_REQUIRED = 200
        if len(state.df_final) < MINIMUM_CANDLES_REQUIRED:
            print(f"\n[WARM-UP] Accumulating history: {len(state.df_final)}/{MINIMUM_CANDLES_REQUIRED} candles collected. Skipping math indicators...", flush=True)
            return

        data = state.df_final.copy()
        try:
            numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'avg']
            for col in numeric_cols:
                data[col] = pd.to_numeric(data[col], errors='coerce').astype(float)

            # FIX 5: recompute avg from OHLC every candle, do not just cast it.
            # Otherwise the new candle keeps avg=NaN -> Log=NaN -> sum_nis[-1]=NaN.
            data['avg'] = (data['open'] + data['high'] + data['low'] + data['close']) / 4.0
            data['avg'] = data['avg'].astype(float)
            data['Log'] = np.log(data['avg'])

            data = ARO2(data)
            data = CCI2(data)
            data = WMA2(data)
            data = RSI2(data)
            data = VWAP2(data)
            data = HA2(data)
            data = TGT(data)
            data = VPRO2(data)
            data = SHADOW(data)

            # Diagnostic: uncomment for one candle to prove Log is clean
            # print(data[['avg','Log']].tail(3))
            data = scan_dataframe_fast_dynamic(data, periods=10, q_lookback=60)
            data['M_R'] = data['market_regime'].apply(lambda x: 1 if x == "NON-STATIONARY" else 0)
            data['M_R_roll'] = data['M_R'].rolling(3).mean()
            # print(data[['Log','sum_nis','market_regime']].tail(3))

            if len(data) <= 100:
                price1 = data['avg'].iloc[0]

            state.df_final = data.copy()

            if 'Trig' in data.columns:
                trig = float(data['Trig'].iloc[-1])
                pro_ = k * trig
                stop_ = k_stop * trig

            if len(data) > l:
                data = data.tail(l)
            state.df_final = data.copy()

            required_features = ['Upper_Shadow_Pct','Lower_Shadow_Pct','Upper_Shadow_MA150','Lower_Shadow_MA165','RSI_160',
                        'MA_84','VWAP_300','CCI_150','AAR_195','Vpro_20','HA_Sinal']
            missing_features = [feat for feat in required_features if feat not in data.columns]
            if not missing_features:
                x_sig = data[required_features].copy()

            if 'x_sig' in locals() and not x_sig.empty:
                try:
                    x_sig_clean = x_sig.dropna()
                    if not x_sig_clean.empty:
                        X_sig_sc_array = my_production_scaler.transform(x_sig_clean)
                        X_sig_sc = pd.DataFrame(X_sig_sc_array, columns=x_sig_clean.columns, index=x_sig_clean.index)
                        latest_sample_ready = X_sig_sc.tail(1).values
                        print("\n[ML INFERENCE] Scaled feature matrix generated successfully using fixed parameters.", flush=True)
                        print(f"[ML INFERENCE] Clean Vector for Model Input:\n{latest_sample_ready}", flush=True)
                        data_pca_array = X_sig_sc_array
                        data_pca = pd.DataFrame(
                            data_pca_array,
                            columns=['PC0', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6','PC7', 'PC8', 'P95', 'PC10'],
                            index=x_sig_clean.index
                        )
                        if 'data_pca' in locals() and not data_pca.empty:
                            try:
                                latest_pca_vector = data_pca.values[-1:]
                                print('latest_pca_vector ', latest_pca_vector)
                                model.eval()
                                data_tensor = torch.tensor(latest_pca_vector, dtype=torch.float32)
                                model_device = next(model.parameters()).device
                                data_tensor = data_tensor.to(model_device)
                                with torch.no_grad():
                                    predictions = model(data_tensor).cpu().numpy().flatten()
                                predicted_scores = predictions
                                y_pred_array = predictions
                                current_signal = float(y_pred_array[0])
                                print('current signal ', current_signal)

                                # FIX 6: do NOT use .loc[candle_time_valid, 'y_pred'].
                                # If the timestamp does not match exactly, .loc INSERTS a new
                                # NaN row that becomes the last row. Use iloc[-1].
                                y_pred_col = state.df_final.columns.get_loc('y_pred') if 'y_pred' in state.df_final.columns else None
                                if y_pred_col is None:
                                    state.df_final['y_pred'] = np.nan
                                    y_pred_col = state.df_final.columns.get_loc('y_pred')
                                state.df_final.iloc[-1, y_pred_col] = current_signal
                                print(f"-> Prediction score saved in y_pred: {current_signal}")
                                print('Number of minutes for mean calculation: ', state.df_final['y_pred'].count())

                                if state.df_final['y_pred'].count() > start_window:
                                    mean_pred = state.df_final['y_pred'].mean()
                                    up = mean_pred + mult_up
                                    do = mean_pred - mult_do
                                if state.df_final['y_pred'].count() >= window:
                                    mean_pred = state.df_final['y_pred'].tail(window).mean()
                                    up = mean_pred + mult_up
                                    do = mean_pred - mult_do
                                print('Current thresholds: ', do, up)

                                print('--- Strategy Metrics (Tail 5) ---')
                                if 'my_p' in state.df_final.columns:
                                    print(state.df_final[['y_pred', 'my_p', 'sig1', 'sig2','pos_count','trade_pr']].tail(5), flush=True)

                                cl = float(data['close'].iloc[-1])
                                print('Last price recorded:', cl)
                                clav = float(data['avg'].iloc[-1])

                                if not state.df_final['trade_pr'].empty and abs(int(state.df_final['my_p'].iloc[-1])) >= 1:
                                    last_valid_entry_price = float(state.df_final['trade_pr'].ffill().iloc[-1])
                                    if last_valid_entry_price > 0:
                                        tr = (cl / last_valid_entry_price) - 1.0
                                        # FIX 6b: write trade_return to last row by position, not by .loc[last_timestamp]
                                        if 'trade_return' not in state.df_final.columns:
                                            state.df_final['trade_return'] = 0.0
                                        state.df_final.iloc[-1, state.df_final.columns.get_loc('trade_return')] = float(tr)
                                        print(f"[RISK SYSTEM] Active trade return (tr): {tr * 100.0:.4f}%", flush=True)
                                    else:
                                        tr = 0.0
                                else:
                                    tr = 0.0
                                    if 'trade_return' not in state.df_final.columns:
                                        state.df_final['trade_return'] = 0.0
                                    state.df_final.iloc[-1, state.df_final.columns.get_loc('trade_return')] = 0.0

                                ############# START TRADING RULES #####################
                                if state.df_final['my_p'].iloc[-2] == 0:
                                    if do < current_signal < up:
                                        state.next_sig = "zero"
                                        print('No signal')
                                    elif current_signal >= up and state.df_final['M_R_roll'].iloc[-1] != 1.0:
                                        state.next_sig = "long_1"
                                        print(f"Sinal LONG gravado com sucesso no candle atual!")
                                    elif current_signal <= do and state.df_final['M_R_roll'].iloc[-1] != 1.0:
                                        state.next_sig = "short_1"
                                        print(f"Sinal SHORT gravado com sucesso no candle atual!")

                                elif state.df_final['my_p'].iloc[-2] >= 1:
                                    if current_signal >= up and state.df_final['my_p'].iloc[-2] < (max_pos) and state.df_final['M_R_roll'].iloc[-1] != 1.0:
                                        state.next_sig = "long_2"
                                        print(f"Sinal LONG gravado com sucesso no candle atual!")
                                    elif state.df_final['pos_count'].iloc[-2] >= max_time and current_signal < up:
                                        state.next_sig = "close"
                                        print(f"Sinal CLOSE gravado com sucesso no candle atual!")
                                    elif current_signal <= do2 and state.df_final['pos_count'].iloc[-2] >= min_time:
                                        state.next_sig = "close"
                                        print(f"Sinal CLOSE gravado com sucesso no candle atual!")
                                    elif tr <= stop_ or tr >= pro_:
                                        state.next_sig = "close"
                                        print(f"Sinal CLOSE gravado com sucesso no candle atual!")
                                    else:
                                        state.next_sig = "hold"
                                        print(f"Sinal HOLD gravado com sucesso no candle atual!")

                                elif state.df_final['my_p'].iloc[-2] <= -1:
                                    if current_signal <= do and state.df_final['my_p'].iloc[-2] > (-max_pos) and state.df_final['M_R_roll'].iloc[-1] != 1.0:
                                        state.next_sig = "short_2"
                                        print(f"Sinal SHORT gravado com sucesso no candle atual!")
                                    elif state.df_final['pos_count'].iloc[-2] >= max_time and current_signal > do:
                                        state.next_sig = "close"
                                        print(f"Sinal CLOSE gravado com sucesso no candle atual!")
                                    elif current_signal >= up2 and state.df_final['pos_count'].iloc[-2] >= min_time:
                                        state.next_sig = "close"
                                        print(f"Sinal CLOSE gravado com sucesso no candle atual!")
                                    elif -tr <= stop_ or -tr >= pro_:
                                        state.next_sig = "close"
                                        print(f"Sinal CLOSE gravado com sucesso no candle atual!")
                                    else:
                                        state.next_sig = "hold"
                                        print(f"Sinal HOLD gravado com sucesso no candle atual!")

                                print('##### next_sig#########', state.next_sig)

                                # FIX 6c: do not use .loc[candle_time_valid, ...] to read back - use iloc.
                                print('Last minute signal:', state.df_final['sig1'].iloc[-1])
                                cl_avg = float(state.df_final['avg'].iloc[-1])
                                pos_price = -float(state.df_final['sig1'].iloc[-1]) * cl_avg

                                try:
                                    realized_re
                                except NameError:
                                    realized_re = 0

                                ############ Printing results #####################
                                if state.df_final['my_p'].iloc[-2] == 0:
                                    # FIX 7: realized from trade_log (never truncated), NOT from df_final tail(200).
                                    # Same formula as yours: pnl = -(sig1*trade_pr) - fee
                                    # Now buys == sells is guaranteed.
                                    pnl_sum = 0.0
                                    for _s, _p in state.trade_log:
                                        pnl_sum += -(_s * _p) - abs(_s * _p * 0.0001)
                                    pct_ret = (pnl_sum / price1) * 100
                                    state.realized_re = pct_ret
                                    print('accumulated % return', state.realized_re)
                                    # FIX 8: ntrades = number of times net position returned to 0.
                                    # Old len(log)//2 gave 3 for 2 scaled trades (should be 2).
                                    _net = 0.0
                                    _closes = 0
                                    for _s, _p in state.trade_log:
                                        _net += _s
                                        if _net == 0:
                                            _closes += 1
                                    state.ntrades = _closes
                                    print('number pf trade', state.ntrades)
                                    print(f"[CHECK] buys/sells net in log: {sum(s for s,_ in state.trade_log)} should be 0 when flat")
                                else:
                                    print('realized % return', state.realized_re)
                                    current_p_level = int(state.df_final['my_p'].iloc[-2])
                                    cl = float(state.df_final['close'].iloc[-1])
                                    valid_entries = state.df_final[state.df_final['trade_pr'] > 0]['trade_pr']
                                    if not valid_entries.empty:
                                        if abs(current_p_level) == 2 and len(valid_entries) >= 2:
                                            adj_price = (float(valid_entries.iloc[-1]) + float(valid_entries.iloc[-2])) / 2.0
                                            trade_pnl_pct = ((cl / adj_price) - 1.0) * 100.0 * (current_p_level)
                                            print(f"[RISK N2] Adjusted Entry Price: {adj_price:.5f} | Scaled Trade P&L: {trade_pnl_pct:.4f}%", flush=True)
                                        elif abs(current_p_level) == 3 and len(valid_entries) >= 3:
                                            adj_price = (float(valid_entries.iloc[-1]) + float(valid_entries.iloc[-2]) + float(valid_entries.iloc[-3])) / 3.0
                                            trade_pnl_pct = ((cl / adj_price) - 1.0) * 100.0 * (current_p_level)
                                            print(f"[RISK N3] Adjusted Entry Price: {adj_price:.5f} | Maximum Trade P&L: {trade_pnl_pct:.4f}%", flush=True)
                                        else:
                                            last_single_entry = float(valid_entries.iloc[-1])
                                            trade_pnl_pct = ((cl / last_single_entry) - 1.0) * 100.0 * (current_p_level)
                                            print(f"[RISK N1] Base Entry Price: {last_single_entry:.5f} | Standard Trade P&L: {trade_pnl_pct:.4f}%", flush=True)
                                        if 'trade_return_pct' not in state.df_final.columns:
                                            state.df_final['trade_return_pct'] = np.nan
                                        state.df_final.iloc[-1, state.df_final.columns.get_loc('trade_return_pct')] = trade_pnl_pct
                                    else:
                                        print("[RISK SYSTEM] No valid execution trade entries detected in historical log matrix.", flush=True)

                            except Exception as keras_error:
                                print(f"\n[KERAS INFERENCE CRASH] Error during neural network forwarding pass: {keras_error}", flush=True)
                    else:
                        print("\n[ML INFERENCE WARNING] x_sig is empty after dropping NaNs. Skipping scaling.", flush=True)
                except Exception as scaling_error:
                    print(f"\n[ML INFERENCE CRASH] Error during scaling pipeline execution: {scaling_error}", flush=True)
            else:
                print("\n[ML INFERENCE WARNING] x_sig is missing or empty. Skipping scaling pipeline.", flush=True)

            print(f"\n=============================================", flush=True)
            print(f"   [CANDLE CLOSED] MINUTE ENTRY #{state.count}   ", flush=True)
            print(f"=============================================", flush=True)
            print(f"[INFO] Global baseline reference price (price1): {price1:.5f}", flush=True)
            print("\nCumulative Trading Matrix (df_final):", flush=True)

        except NameError as e:
            print(f"\n[PIPELINE WARNING] Missing technical indicator definition: {e}", flush=True)
            print("Please declare all required indicator functions before initiating the stream.", flush=True)
            return

        print(df_tick, flush=True)

    except Exception as pipeline_error:
        import traceback
        print(f"[PIPELINE CRASH] Error: {pipeline_error}", flush=True)
        traceback.print_exc(file=sys.stdout)
        sys.stdout.flush()


if __name__ == "__main__":
    FALLBACK_URL = "wss://stream.binance.com/ws/ethusdt@kline_1m"
    print(f"Connecting to Binance Fallback Endpoint: {FALLBACK_URL}", flush=True)
    while True:
        try:
            ws = websocket.create_connection(
                FALLBACK_URL,
                timeout=10,
                suppress_origin=True
            )
            print("### Connection successfully open and authenticated! ###", flush=True)
            print("Streaming data now...", flush=True)
            while True:
                message = ws.recv()
                process_pipeline_sync(message)
        except websocket.WebSocketTimeoutException:
            print("[WARNING] Connection timeout. Re-pinging server...", flush=True)
            continue
        except KeyboardInterrupt:
            print("\nClosing WebSocket connection...", flush=True)
            if 'ws' in locals(): ws.close()
            print("Disconnected.", flush=True)
            break
        except Exception as e:
            print(f"### Connection lost ({e}). Reconnecting in 5 seconds... ###", flush=True)
            time.sleep(5)

Wiping old lingering WebSocket connection from memory...
Initializing clean WebSocket connection tunnel...
Executing emergency resource cleanup. Unlinking network sockets...
Success: System state restored. Ready for re-run without kernel restart!
### Production Scaler successfully loaded into memory! ###
Connecting to Binance Fallback Endpoint: wss://stream.binance.com/ws/ethusdt@kline_1m
### Connection successfully open and authenticated! ###
Streaming data now...

[CONFIRMED] CANDLE SAVED AT: 2026-11-16 12:16:00 | Rows: 1

[WARM-UP] Accumulating history: 1/200 candles collected. Skipping math indicators...

[CONFIRMED] CANDLE SAVED AT: 2026-11-16 12:17:00 | Rows: 2

[WARM-UP] Accumulating history: 2/200 candles collected. Skipping math indicators...

[CONFIRMED] CANDLE SAVED AT: 2026-11-16 12:18:00 | Rows: 3

[WARM-UP] Accumulating history: 3/200 candles collected. Skipping math indicators...
